In [4]:
import math
from math import sqrt, log
from utils import *
import networkx as nx
from scipy.spatial.distance import euclidean
import numpy as np
from read_arbor_reconstruction import read_arbor_full, read_arbor_full_initial
from constants import *
from optimal_midpoint import optimal_midpoint, optimal_midpoint_approx, optimal_midpoint_alpha1
from collections import defaultdict, namedtuple
import seaborn as sns
import os
import pandas as pd
from scipy.optimize import minimize_scalar, fsolve
import pareto_functions as pf

import plotly.graph_objs as go # for plotting arbors
import plant_gravitropism as pg



CostSpec = namedtuple('CostSpec', ['wiring_transform', 'delay_transform'])
"""
VVVVVVVV originals VVVVVVVVV

def _homogeneous_wiring(curve, to_root): return curve
def _homogeneous_delay(curve, to_root): return curve + to_root

def _heterogeneous_wiring(curve, to_root): return curve ** 2
def _heterogeneous_delay(curve, to_root): return log(1 + curve) + log(1 + to_root)

"""
def _homogeneous_wiring(curve, to_root): return curve
def _homogeneous_delay(curve, to_root): 
    if (curve + to_root) < 0: 
        print("Delay is negative for HOMOGENEOUS") 
    return curve + to_root

def _heterogeneous_wiring(curve, to_root): return curve ** 2
def _heterogeneous_delay(curve, to_root): 
    if (log(1 + curve) + log(1 + to_root)) < 0: 
        print("Delay is negative for HOMOGENEOUS") 
    return log(1 + curve) + log(1 + to_root)

HOMOGENEOUS = CostSpec(
    wiring_transform = _homogeneous_wiring,
    delay_transform = _homogeneous_delay,
)

HETEROGENEOUS = CostSpec(
    wiring_transform = _heterogeneous_wiring, 
    delay_transform = _heterogeneous_delay,
)

COST_SPECS = {
    'homogeneous': HOMOGENEOUS,
    'heterogeneous': HETEROGENEOUS,
}





# Calculates length of the lateral root
def lateral_root_path_length(G, tip):
    """Sum edge lengths from tip back to main root insertion point."""
    length = 0
    visited = set()
    queue = [tip]
    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                label = G.nodes[neighbor]['label']
                if label in ('lateral root', 'lateral root tip'):
                    length += G[node][neighbor]['length']
                    queue.append(neighbor)
                elif label in ('main root', 'main root base'):
                    length += G[node][neighbor]['length']
                    # stop here — this is the insertion point
    return length

# note: lateral_root_path_length requires a tip, which seems to be an object that can be 
# converted into a queue that is later traversed (elements are popped).

# Claude-generated conduction_delay function
def conduction_delay(G, cost_spec=HOMOGENEOUS):
    print("CONDUCTION DELAY")
    droot = {}
    queue = []
    visited = set()
    root = G.graph.get('main root base', G.graph.get('main root'))
    queue.append(root)
    droot[root] = 0
    delay = 0

    while len(queue) > 0:
        curr = queue.pop(0)
        assert curr not in visited
        visited.add(curr)

        if G.nodes[curr]['label'] == 'lateral root tip':
            curve = lateral_root_path_length(G, curr)
            to_root = droot[curr] - curve   # subtract lateral length to get main root distance
            delay += cost_spec.delay_transform(curve, to_root)

        for u in G.neighbors(curr):
            if u not in visited:
                queue.append(u)
                droot[u] = droot[curr] + G[curr][u]['length']

    assert len(visited) == G.number_of_nodes()
    return delay







# from plant_gravitropism

# ---- evaluate_parameters ----

# -- arbor_best_cost
def compute_main_root_base_distances(arbor):
    """
    Returns a dict mapping each main root node to its distance from the main root base,
    using edge 'length' attributes and the ordering from get_main_root_segments.
    """
    base = arbor.graph['main root base']

    base_dist = {base: 0}
    for seg_start, seg_end in get_main_root_segments(arbor):
        base_dist[seg_end] = base_dist[seg_start] + arbor[seg_start][seg_end]['length']

    return base_dist

def get_insertion_segment(arbor, lateral_tip, segments):
    """
    For a given lateral root tip, walk up the graph until hitting a main root node,
    then find which segment that insertion point belongs to.
    Returns all segments up to and including the insertion segment.

    Parameters
    ----------
    arbor : networkx.Graph
        Observed arbor graph.
    lateral_tip : tuple
        (x, y) of the lateral root tip.
    segments : list
        Ordered list of main root segments from get_main_root_segments.

    Returns
    -------
    list of segments up to and including the insertion segment
    """
    # BFS up from tip until we hit a main root node
    visited = set()
    queue = [lateral_tip]
    insertion_point = None

    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        label = arbor.nodes[node]['label']
        if label in ('main root', 'main root base'):
            insertion_point = node
            break
        for neighbor in arbor.neighbors(node):
            if neighbor not in visited:
                queue.append(neighbor)

    assert insertion_point is not None, f"No main root node found from tip {lateral_tip}"

    # Find the segment that contains the insertion point and return all up to it
    valid_segments = []
    for seg in segments:
        valid_segments.append(seg)
        if insertion_point in seg:
            break

    return valid_segments

# - optimize_tip - 
def is_between(a, x, b):
    """Returns True if x is strictly between a and b (in either order)."""
    return a < x < b or b < x < a

# find_best_cost_brute_force
def branch_point_from_t(x0, y0, x1, y1, t):
    """Linearly interpolate a branch point along segment (x0,y0)-(x1,y1) at parameter t."""
    return x0 + t * (x1 - x0), y0 + t * (y1 - y0)

# ... compute_cost ...
def curve_length(G, x0, y0, p, q):
    """
    Arc length of the parabola G*x^2 + b*x + c between (x0, y0) and (p, q).
    Uses closed-form solution when G != 0, Euclidean distance when G == 0.
    """
    if G == 0:
        # Straight line distance
        return euclidean((x0, y0), (p, q))

    # Shift to local frame: branch point becomes origin
    p_local = p - x0
    q_local = q - y0

    k = (q_local - G * p_local**2) / p_local

    theta_0 = math.atan(k)
    theta_p = math.atan(2 * G * p_local + k)

    def sec(theta):
        return 1.0 / math.cos(theta)

    def F(theta):
        return sec(theta) * math.tan(theta) + math.log(abs(sec(theta) + math.tan(theta)))

    return abs((1.0 / (4 * G)) * (F(theta_p) - F(theta_0)))


def compute_cost(alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, tip_x, tip_y, cost_spec=HOMOGENEOUS):
    """
    Compute total cost, wiring, and delay for a given branch point.

    Parameters
    ----------
    seg_base_dist : float
        Distance from root base to start of segment (x0, y0).
    t : float
        Position along segment [0, 1].
    seg_length : float
        Length of the main root segment.
    branch_x, branch_y : float
        Coordinates of the branch point on the main root.
    tip_x, tip_y : float
        Coordinates of the lateral root tip.
    """
    curve = curve_length(G, branch_x, branch_y, tip_x, tip_y)
    to_root = seg_base_dist + t * seg_length
    wiring = cost_spec.wiring_transform(curve, to_root)
    delay = cost_spec.delay_transform(curve, to_root)
    cost = alpha * wiring + (1 - alpha) * delay
    return cost, wiring, delay


def find_best_cost_brute_force(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=HOMOGENEOUS):
    """
    Find the optimal branch point on segment (x0,y0)-(x1,y1) for lateral tip (p, q)
    by brute-force search over t in [0, 1] with step 0.01.

    Returns
    -------
    tuple : (cost, wiring, delay, best_t, best_x, best_y, p, q)
    """
    seg_length = euclidean((x0, y0), (x1, y1))
    best_cost = math.inf
    best_wiring = math.inf
    best_delay = math.inf
    best_t = None
    best_x = None
    best_y = None

    for t in pylab.arange(0, 1 + 0.01, 0.01):
        branch_x, branch_y = branch_point_from_t(x0, y0, x1, y1, t)
        cost, wiring, delay = compute_cost(
            alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, p, q, cost_spec=cost_spec 
        )
        if cost <= best_cost:
            best_cost = cost
            best_wiring = wiring
            best_delay = delay
            best_t = t
            best_x = branch_x
            best_y = branch_y

    return best_cost, best_wiring, best_delay, best_t, best_x, best_y, p, q


def find_best_cost_brent(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=HOMOGENEOUS):
    """
    Find the optimal branch point on segment (x0,y0)-(x1,y1) for lateral tip (p, q)
    using Brent's method to directly minimize the cost function.

    For G=0, falls back to exact analytical solution from optimal_midpoint.py.
    For G!=0, uses minimize_scalar with method='bounded' on [0, 1].

    Returns
    -------
    tuple : (cost, wiring, delay, best_t, best_x, best_y, p, q)
    """
    seg_length = euclidean((x0, y0), (x1, y1))

    # G != 0: minimize cost directly using Brent's method
    def cost_at_t(t):
        branch_x, branch_y = branch_point_from_t(x0, y0, x1, y1, t)
        c, _, _ = compute_cost(alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, p, q, cost_spec=cost_spec)
        return c

    result = minimize_scalar(cost_at_t, bounds=(0, 1), method='bounded')
    best_t = result.x
    best_x, best_y = branch_point_from_t(x0, y0, x1, y1, best_t)
    best_cost, best_wiring, best_delay = compute_cost(
        alpha, G, seg_base_dist, best_t, seg_length, best_x, best_y, p, q, cost_spec=cost_spec
    )

    return best_cost, best_wiring, best_delay, best_t, best_x, best_y, p, q


# find_best_cost_analytical
def make_costprime(G, alpha, l, theta, p, q):
    """
    Returns the derivative of cost w.r.t. t for the analytical case.

    Parameters
    ----------
    G : float
        Gravity parameter.
    alpha : float
        Weighting parameter.
    l : float
        Length of the main root segment.
    theta : float
        Angle of the segment.
    p, q : float
        Coordinates of the lateral root tip (shifted to local frame).
    """
    A1 = G * (l * math.cos(theta))**2
    B1 = -l * math.sin(theta)
    C1 = q - G * p**2
    D1 = -l * math.cos(theta)
    E1 = p

    def b(t):
        return (A1*t**2 + B1*t + C1) / (D1*t + E1)

    def bprime(t):
        num = (D1*t + E1) * (2*A1*t + B1) - D1 * (A1*t**2 + B1*t + C1)
        den = (D1*t + E1)**2
        return num / den

    def costprime(t):
        bt = b(t)
        term1 = (bprime(t) / (2 * G)) * (
            math.sqrt(1 + (2*G*p + bt)**2) -
            math.sqrt(1 + (2*G * t * l * math.cos(theta) + bt)**2)
        )
        term2 = (1 - alpha) * l
        term3 = math.sqrt(1 + (2*G * t * l * math.cos(theta) + bt)**2) * l * math.cos(theta)
        return term1 + term2 - term3

    return costprime

def find_root_in_unit_interval(func, num_guesses=1):
    """
    Find roots of func in [0, 1] using fsolve with evenly spaced initial guesses,
    excluding endpoints.
    """
    roots = []
    guesses = np.linspace(0, 1, num_guesses + 2)[1:-1]
    for guess in guesses:
        try:
            root = fsolve(func, guess)[0]
            if 0 <= root <= 1 and not any(np.isclose(root, r) for r in roots):
                roots.append(root)
        except Exception:
            pass
    return sorted(roots)

def find_best_cost_analytical(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=HOMOGENEOUS):
    """
    Find the optimal branch point on segment (x0,y0)-(x1,y1) for lateral tip (p, q)
    using the analytical derivative of the cost function.

    For G=0, uses exact analytical solution from optimal_midpoint.py.
    For alpha=1, minimizes curve length directly via scipy minimize_scalar.
    Otherwise, uses analytical costprime approach with fsolve.

    Returns
    -------
    tuple : (cost, wiring, delay, best_t, best_x, best_y, p, q)
    """

    seg_length = euclidean((x0, y0), (x1, y1))

    # G=0 case: use exact analytical solution from optimal_midpoint.py
    if G == 0:
        if alpha == 1:
            cost_val, (best_x, best_y), best_t = optimal_midpoint.optimal_midpoint_alpha1(
                (x0, y0), (x1, y1), (p, q)
            )
        else:
            cost_val, (best_x, best_y), best_t = optimal_midpoint.optimal_midpoint_exact(
                (x0, y0), (x1, y1), (p, q), alpha, seg_base_dist
            )
        wiring = euclidean((best_x, best_y), (p, q))
        to_root = seg_base_dist + best_t * seg_length
        delay = wiring + to_root
        return cost_val, wiring, delay, best_t, best_x, best_y, p, q
    # G != 0: use analytical costprime with fsolve
    else:
        # theta = math.atan2(abs(y1 - y0), abs(x1 - x0)) if (x1 != x0 and y1 != y0) else 0
        theta = math.atan2(y1 - y0, x1 - x0)
        p_local = p - x0
        q_local = q - y0

        costprime = make_costprime(G, alpha, seg_length, theta, p_local, q_local)

        def cost_at_t(t):
            branch_x, branch_y = branch_point_from_t(x0, y0, x1, y1, t)
            c, _, _ = compute_cost(alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, p, q, cost_spec=cost_spec)
            return c

        roots = find_root_in_unit_interval(costprime)
        valid_roots = [r for r in roots if 0 <= r <= 1]

        if valid_roots:
            best_t = min(valid_roots, key=cost_at_t)
        else:
            best_t = 0.0 if cost_at_t(0) <= cost_at_t(1) else 1.0

        best_x, best_y = branch_point_from_t(x0, y0, x1, y1, best_t)
        best_cost, best_wiring, best_delay = compute_cost(
            alpha, G, seg_base_dist, best_t, seg_length, best_x, best_y, p, q, cost_spec=cost_spec
        )

        return best_cost, best_wiring, best_delay, best_t, best_x, best_y, p, q


OPTIMIZATION_METHOD = 'brent'
def optimize_tip(tip, segments, base_dist, alpha, G, cost_spec=HOMOGENEOUS):
    p, q = tip
    results = []

    for seg in segments:
        x0, y0 = seg[0]
        x1, y1 = seg[1]
        seg_base_dist = base_dist[(x0, y0)]

        if is_between(x0, p, x1) or OPTIMIZATION_METHOD == 'brute_force':
            result = find_best_cost_brute_force(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=cost_spec)
        elif OPTIMIZATION_METHOD == 'brent':
            result = find_best_cost_brent(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=cost_spec)
        else:
            result = find_best_cost_analytical(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=cost_spec)

        results.append(result)

    best = min(results)
    return best

def arbor_best_cost(arbor, G, alpha, cost_spec=HOMOGENEOUS):
    """
    For each lateral root tip in the arbor, find the optimal branch point
    on the main root under the given (G, alpha) parameters.

    Parameters
    ----------
    arbor : networkx.Graph
        Already-loaded observed arbor graph.
    G : float
        Gravity parameter.
    alpha : float
        Weighting parameter.

    Returns
    -------
    list of tuples : [(cost, wiring, delay, best_t, best_x, best_y, tip_x, tip_y), ...]
    """
    segments = get_main_root_segments(arbor)
    base_dist = compute_main_root_base_distances(arbor)

    lat_tips = [
        node for node in arbor.nodes()
        if arbor.nodes[node]['label'] == 'lateral root tip'
    ]

    final = []
    for tip in lat_tips:
        valid_segments = get_insertion_segment(arbor, tip, segments)
        result = optimize_tip(tip, valid_segments, base_dist, alpha, G, cost_spec=cost_spec)
        if result is not None:
            final.append(result)
        else:
            print(f"Warning: No valid results for lateral tip at {tip}")

    return final


# calculate_orthogonal_errors

def collect_lateral_root_points(arbor, lateral_tip):
    """
    BFS from lateral_tip through 'lateral root' and 'lateral root tip' nodes.
    """
    lateral_points = []
    visited = set()
    queue = [lateral_tip]

    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        label = arbor.nodes[node]['label']
        if label in ('lateral root', 'lateral root tip'):
            lateral_points.append(node)
            for neighbor in arbor.neighbors(node):
                if neighbor not in visited and arbor.nodes[neighbor]['label'] in ('lateral root', 'lateral root tip'):
                    queue.append(neighbor)

    return lateral_points


def calc_coeff(G, x, y, p, q):
    """
    Returns coefficients b, c of the parabola G*x^2 + b*x + c
    that passes through (x, y) and (p, q).
    """
    b = (q - y - G * (p*p - x*x)) / (p - x)
    c = q - G * p*p - b * p
    return b, c

def collect_lateral_root_segments(arbor, lateral_tip):
    """
    Return list of (x0, y0, x1, y1) segments along the lateral root path
    from tip back to the main root insertion point.
    """
    segments = []
    path = collect_lateral_root_points(arbor, lateral_tip)  # existing BFS
    for i in range(len(path) - 1):
        x0, y0 = path[i]
        x1, y1 = path[i + 1]
        segments.append((x0, y0, x1, y1))
    return segments

def calculate_orthogonal_errors(gravity, arbor, main_root_pt, lateral_tip,
                                 n_subsample=100):
    """
    Compute total orthogonal distance and total squared orthogonal distance
    between the fitted parabola and the lateral root path.

    Sub-discretizes each lateral root segment into n_subsample points
    so that lightly-traced lateral roots are treated consistently with
    densely-traced ones.
    """
    px, py = main_root_pt
    tip_x, tip_y = lateral_tip

    b, c = calc_coeff(gravity, px, py, tip_x, tip_y)
    x_start, x_end = min(px, tip_x), max(px, tip_x)

    segments = collect_lateral_root_segments(arbor, lateral_tip)

    if not segments:
        return 0.0, 0.0

    # Sub-discretize all segments into sample points — vectorized
    all_xs = []
    all_ys = []
    for x0, y0, x1, y1 in segments:
        ts = np.linspace(0, 1, n_subsample, endpoint=False)
        all_xs.append(x0 + ts * (x1 - x0))
        all_ys.append(y0 + ts * (y1 - y0))

    # Include the final endpoint of the last segment
    all_xs.append(np.array([segments[-1][2]]))
    all_ys.append(np.array([segments[-1][3]]))

    obs_x = np.concatenate(all_xs)
    obs_y = np.concatenate(all_ys)

    # Vectorized orthogonal distance to parabola
    # Find closest point on parabola G*x^2 + b*x + c to each (obs_x, obs_y)
    xs = np.linspace(x_start, x_end, 1000)
    ys = gravity * xs**2 + b * xs + c

    # For each observed point, find minimum distance to any point on the curve
    # Shape: (n_observed, 1) - (1, n_curve) = (n_observed, n_curve)
    dx = obs_x[:, np.newaxis] - xs[np.newaxis, :]
    dy = obs_y[:, np.newaxis] - ys[np.newaxis, :]
    dists = np.sqrt(dx**2 + dy**2).min(axis=1)

    total_orthogonal = dists.sum()
    total_sq_orthogonal = (dists**2).sum()

    return total_orthogonal, total_sq_orthogonal



def main_root_length(arbor):
    """Total length of the main root, using edge 'length' attributes."""
    return sum(
        arbor[u][v]['length']
        for u, v in get_main_root_segments(arbor)
    )



def evaluate_parameters(arbor, G, alpha, cost_spec=HOMOGENEOUS):
    """
    Evaluate a single (G, alpha) combination for an already-loaded arbor graph.

    Returns
    -------
    tuple : (wiring, delay, total_orthogonal, total_sq_orthogonal)
    """
    results = arbor_best_cost(arbor, G, alpha, cost_spec=cost_spec)

    wiring = 0
    delay = 0
    total_orthogonal = 0
    total_sq_orthogonal = 0

    for result in results:
        wiring += result[1]
        delay += result[2]

        main_root_pt = (result[4], result[5])
        lateral_tip = (result[6], result[7])

        orth, sq_orth = calculate_orthogonal_errors(G, arbor, main_root_pt, lateral_tip)
        total_orthogonal += orth
        total_sq_orthogonal += sq_orth

    wiring += main_root_length(arbor)

    return wiring, delay, total_orthogonal, total_sq_orthogonal


# get_main_root_segments

def get_main_root_segments(arbor):
    """
    BFS from main root base along 'main root' labeled nodes, returning
    an ordered list of segment tuples ((x0, y0), (x1, y1)).
    """
    base = arbor.graph['main root base']

    segments = []
    visited = {base}
    queue = [base]

    while queue:
        curr = queue.pop(0)
        for neighbor in arbor.neighbors(curr):
            if neighbor in visited:
                continue
            if arbor.nodes[neighbor]['label'] in ('main root', 'main root base'):
                segments.append((curr, neighbor))
                visited.add(neighbor)
                queue.append(neighbor)

    return segments








# from pareto_functions.py 


def get_line_segment_drawings(line_segments, color="gray"):
    """Convert line segments into Plotly Scatter objects."""
    traces = []
    for seg in line_segments.values():
        (x0, y0), (x1, y1) = seg  # node IDs are tuples
        traces.append(go.Scatter(
            x=[x0, x1], y=[y0, y1],
            mode="lines",
            line=dict(color=color, width=4),
            showlegend=False
        ))
    return traces

def get_observed_lateral_segments(arbor):
    """
    Trace all lateral roots from tip to main root.
    Nodes are tuples (x, y).
    """
    segments = []
    tips = [n for n in arbor.nodes if arbor.nodes[n]["label"] == "lateral root tip"]

    for tip in tips:
        curr = tip
        prev = None
        while True:
            neighbors = list(arbor.neighbors(curr))
            if prev is not None:
                neighbors = [n for n in neighbors if n != prev]

            if len(neighbors) == 0:
                break  # end of lateral root

            next_node = neighbors[0]
            segments.append((curr, next_node))
            prev, curr = curr, next_node

            if arbor.nodes[curr]["label"].startswith("main root"):
                break

    return segments

def get_lateral_insertion_points(lateral_segments, arbor):
    """
    Find the coordinates where each lateral root meets the main root.
 
    These are the endpoints of lateral segments whose terminal node carries
    a 'main root' or 'main root base' label — i.e. the last node reached
    when tracing a lateral from its tip toward the main stem.
 
    Returns a list of (x, y) coordinate tuples, one per lateral root that
    successfully reaches the main root. Duplicate insertion points (two
    laterals sharing one junction) are included only once.

    (generated by Claude)
    """
    insertion_points = set()
    for seg in lateral_segments:
        node_a, node_b = seg
        if arbor.nodes[node_b]["label"].startswith("main root"):
            insertion_points.add(node_b)
    return list(insertion_points)

def get_lateral_segment_drawings(lateral_segments, color="lightgray"):
    traces = []
    for seg in lateral_segments:
        (x0, y0), (x1, y1) = seg
        traces.append(go.Scatter(
            x=[x0, x1], y=[y0, y1],
            mode="lines",
            line=dict(color=color, width=2),
            showlegend=False
        ))
    return traces

def get_tip_drawings(lat_tips, color="orange"):
    return [go.Scatter(x=[tip[0]], y=[tip[1]],
                       mode="markers",
                       marker=dict(color=color),
                       showlegend=False)
            for tip in lat_tips]

def get_insertion_point_drawings(insertion_points, color="blue"):
    """
    Draw lateral root insertion points (where a lateral meets the main root)
    as blue markers the same size as the lateral root tip markers.

    (generated by Claude)
    """
    return [go.Scatter(
                x=[pt[0]], y=[pt[1]],
                mode="markers",
                marker=dict(color=color, size=8),
                showlegend=False)
            for pt in insertion_points]

def get_lateral_nodes(lateral_segments):
    """
    Collect every unique node that appears anywhere along the lateral root
    segments — both endpoints of every segment.
 
    This covers all intermediate nodes along each lateral (the nodes between
    the tip and the insertion point), as well as the tips and insertion points
    themselves. Duplicates are removed so overlapping segment endpoints are
    not drawn twice.

    (generated by Claude)
    """
    nodes = set()
    for node_a, node_b in lateral_segments:
        nodes.add(node_a)
        nodes.add(node_b)
    return list(nodes)

def get_lateral_node_drawings(lateral_nodes, color="white"):
    """
    Draw every node along the lateral root segments as a default-sized marker.
    No explicit size is passed so Plotly uses its own default (6px).

    (generated by Claude)
    """
    return [go.Scatter(
                x=[node[0]], y=[node[1]],
                mode="markers",
                marker=dict(color=color),
                showlegend=False)
            for node in lateral_nodes]


def create_graphs(arbor, G, alpha):
    """
        Changed slightly for toy network
    """
    return arbor

def plot_arbors(arbor, G, alpha, show_observed=True, paper=False, save_fname=None):
    arbor_name = arbor.graph.get('arbor name', 'toy arbor')

    wiring, delay, total_orthogonal, total_sq_orthogonal = evaluate_parameters(arbor, G, alpha)

    print(f"\n→ G = {G}, alpha = {alpha}")
    print(f"Wiring cost: {wiring:.4f}")
    print(f"Conduction delay: {delay:.4f}\n")
    print(f"Total orthogonal distance: {total_orthogonal:.4f}")
    print(f"Total squared orthogonal distance: {total_sq_orthogonal:.4f}\n")

    fig = go.Figure()

    if show_observed:
        main_segments = get_main_root_segments(arbor)
        # convert list of tuples to dict for get_line_segment_drawings
        main_segments_dict = {i: seg for i, seg in enumerate(main_segments)}
        for trace in get_line_segment_drawings(main_segments_dict, color="black"):
            fig.add_trace(trace)

        lateral_segments = get_observed_lateral_segments(arbor)
        for trace in get_lateral_segment_drawings(lateral_segments, color="green"):
            fig.add_trace(trace)

        # --- all nodes along lateral root segments (white, default size) ---
        lateral_nodes = get_lateral_nodes(lateral_segments)
        for trace in get_lateral_node_drawings(lateral_nodes, color="white"):
            fig.add_trace(trace)

        lat_tips = [n for n in arbor.nodes if arbor.nodes[n]["label"] == "lateral root tip"]
        for trace in get_tip_drawings(lat_tips, color="orange"):
            fig.add_trace(trace)
        
        # --- lateral root insertion points (blue, same size as tips) ---
        insertion_points = get_lateral_insertion_points(lateral_segments, arbor)
        for trace in get_insertion_point_drawings(insertion_points, color="blue"):
            fig.add_trace(trace)

        base_node = arbor.graph['main root base']
        fig.add_trace(go.Scatter(
            x=[base_node[0]], y=[base_node[1]],
            mode="markers",
            marker=dict(color="purple", size=30),
            name="Main root base"
        ))

    fig.update_layout(
        title=f"Arbor: {arbor_name}   |   G={G}, alpha={alpha}",
        annotations=[
            dict(
                text="",
                xref="paper", yref="paper",
                x=0.5, y=-0.1,
                showarrow=False,
                font=dict(size=14)
            )
        ],
        xaxis_title="X",
        yaxis_title="Y",
        yaxis_autorange="reversed",
        width=850,
        height=700,
        margin=dict(t=80, b=80)
    )
    
    if paper:
        fig.update_layout(
            xaxis_title=None,
            yaxis_title=None,
            annotations=[], 
            title_text="", 
            showlegend=False
        )
        fig.update_xaxes(showticklabels=False)
        fig.update_yaxes(showticklabels=False)


    fig.show()
    
    if save_fname != None:
        print("saving fig to " + save_fname)
        fig.write_image(save_fname)



# Function that allows customizable test arbors 
def toy_arbor_gen(root, laterals, name='toy arbor'):
    root_nodes = list(root)
    if len(root_nodes) == 0:
        raise ValueError('root must contain at least one coordinate')

    G = nx.Graph()
    root_base = root_nodes[0]

    for r in root_nodes:
        G.add_node(r)
        G.nodes[r]['label'] = 'main root'

    G.nodes[root_base]['label'] = 'main root base'
    G.graph['main root base'] = root_base
    G.graph['arbor name'] = name

    for u, v in zip(root_nodes, root_nodes[1:]):
        connect_points(G, u, v)

    for lateral in laterals:
        G.add_node(lateral)
        G.nodes[lateral]['label'] = 'lateral root'
        connect_points(G, root_base, lateral)

    relabel_lateral_root_tips(G)

    return G

# Run above ^^

In [8]:
plot_arbors("pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt006_M248_5_C_10aba", 0, .1)

AttributeError: 'str' object has no attribute 'graph'

In [1]:
import draw_arbors_AM as draw
import math
import networkx as nx
import numpy as np
import plotly.graph_objs as go
import pylab
import plant_gravitropism as pg
import read_arbor_reconstruction as rar


In [7]:
# Conduction delay calculated from a CSV file
fnames = ["pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt006_M248_5_C_10aba.csv", 
          
          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt009_M248_2_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv",

            "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt058_M248_3_S_1aba.csv",

            "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt059_M248_2_S_1aba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt061_M248_7_C_1aba.csv",
          ]


# ---- Checking if there are visual differences between arbors with (-) to_root values
for fname in fnames:
    arbor = rar.read_arbor_full_initial(fname)
    plot_arbors(fname, 0, .1)





AttributeError: 'str' object has no attribute 'graph'